# 03. Data Preprocessing Pipeline

This notebook creates a reusable preprocessing pipeline that:
1. Handles missing values
2. Creates engineered features
3. Scales and encodes data
4. Saves the preprocessing pipeline for production use

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Load data
df = pd.read_csv('../data/raw/synthetic_students.csv')
print(f"Loaded {len(df)} records")
df.head()

Loaded 500 records


,student_id,student_name,attendance_percentage,internal_marks,assignment_submission_rate,semester,risk_score,risk_level
0,STU0001,Allison Hill,84.9,98.5,66.5,2,28.55,Low
1,STU0002,Noah Rhodes,70.3,100.0,64.1,1,26.72,Low
2,STU0003,Angie Henderson,65.6,79.4,83.6,5,20.61,Low
3,STU0004,Daniel Wagner,79.8,52.5,22.2,4,37.00,Moderate
4,STU0005,Cristian Santos,54.7,68.4,77.9,4,21.06,Low


## 1. Feature Engineering

Create new features that capture domain knowledge about student risk.

In [3]:
def create_features(df):
    """
    Create engineered features from raw data.
    
    Features created:
    - engagement_score: Combined metric of attendance + assignment
    - performance_index: Normalized academic performance
    - risk_flags: Binary indicators for concerning patterns
    """
    df = df.copy()
    
    # Feature 1: Engagement Score (weighted combination)
    df['engagement_score'] = (
        df['attendance_percentage'] * 0.6 +
        df['assignment_submission_rate'] * 0.4
    )
    
    # Feature 2: Performance Index (marks relative to engagement)
    # Students performing below their engagement level may need help
    df['performance_index'] = df['internal_marks'] - df['engagement_score']
    
    # Feature 3: Binary risk flags
    df['low_attendance_flag'] = (df['attendance_percentage'] < 60).astype(int)
    df['low_marks_flag'] = (df['internal_marks'] < 50).astype(int)
    df['low_assignment_flag'] = (df['assignment_submission_rate'] < 50).astype(int)
    
    # Feature 4: Combined risk flag (multiple indicators)
    df['multiple_risk_flags'] = (
        df['low_attendance_flag'] +
        df['low_marks_flag'] +
        df['low_assignment_flag']
    )
    
    # Feature 5: Semester group (early, mid, late)
    def group_semester(sem):
        if sem <= 2:
            return 'early'
        elif sem <= 5:
            return 'mid'
        else:
            return 'late'
    df['semester_group'] = df['semester'].apply(group_semester)
    
    return df

# Apply feature engineering
df_engineered = create_features(df)
print("Features created:")
print(df_engineered.columns.tolist())

Features created:
['student_id', 'student_name', 'attendance_percentage', 'internal_marks', 'assignment_submission_rate', 'semester', 'risk_score', 'risk_level', 'engagement_score', 'performance_index', 'low_attendance_flag', 'low_marks_flag', 'low_assignment_flag', 'multiple_risk_flags', 'semester_group']


In [4]:
# Show engineered features
print("\nEngineered Features Preview:")
cols = ['engagement_score', 'performance_index', 'low_attendance_flag', 
        'low_marks_flag', 'low_assignment_flag', 'multiple_risk_flags']
df_engineered[cols].head(10)


Engineered Features Preview:


,engagement_score,performance_index,low_attendance_flag,low_marks_flag,low_assignment_flag,multiple_risk_flags
0,77.54,20.96,0,0,0,0
1,67.82,32.18,0,0,0,0
2,72.80,6.60,0,0,0,0
3,56.76,-4.26,0,0,1,1
4,63.98,4.42,1,0,0,1
5,85.76,8.14,0,0,0,0
6,67.58,-1.08,0,0,0,0
7,62.88,9.52,0,0,0,0
8,62.26,31.24,0,0,1,1
9,55.92,2.08,0,0,1,1


## 2. Preprocessing Pipeline

In [5]:
# Define feature sets
base_features = ['attendance_percentage', 'internal_marks', 'assignment_submission_rate', 'semester']
engineered_features = ['engagement_score', 'performance_index', 'multiple_risk_flags']
all_features = base_features + engineered_features

# Prepare features and target
X = df_engineered[all_features].copy()
y = df_engineered['risk_level'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {all_features}")

Feature matrix shape: (500, 7)
Features: ['attendance_percentage', 'internal_marks', 'assignment_submission_rate', 'semester', 'engagement_score', 'performance_index', 'multiple_risk_flags']


In [6]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), all_features)
    ])

# Fit the preprocessor
X_processed = preprocessor.fit_transform(X)

print("\nPreprocessing complete!")
print(f"Processed shape: {X_processed.shape}")
print(f"Mean (should be ~0): {X_processed.mean(axis=0).round(4)}")
print(f"Std (should be ~1): {X_processed.std(axis=0).round(4)}")


Preprocessing complete!
Processed shape: (500, 7)
Mean (should be ~0): [-0.  0. -0. -0. -0. -0.  0.]
Std (should be ~1): [1. 1. 1. 1. 1. 1. 1.]


In [7]:
# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel encoding:")
print(f"Classes: {list(label_encoder.classes_)}")
print(f"Encoding mapping: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")


Label encoding:
Classes: ['High', 'Low', 'Moderate']
Encoding mapping: {'High': 0, 'Low': 1, 'Moderate': 2}


## 3. Train-Test Split

In [8]:
# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

# Check stratification
print(f"\nOriginal distribution: {pd.Series(y_encoded).value_counts().to_dict()}")
print(f"Train distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test distribution: {pd.Series(y_test).value_counts().to_dict()}")

Training set: 400 samples (80.0%)
Test set: 100 samples (20.0%)

Original distribution: {1: 400, 2: 92, 0: 8}
Train distribution: {1: 320, 2: 74, 0: 6}
Test distribution: {1: 80, 2: 18, 0: 2}


## 4. Save Preprocessing Artifacts

In [9]:
# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor.joblib')
print("Saved: ../models/preprocessor.joblib")

# Save label encoder
joblib.dump(label_encoder, '../models/label_encoder.joblib')
print("Saved: ../models/label_encoder.joblib")

# Save feature information
feature_config = {
    'base_features': base_features,
    'engineered_features': engineered_features,
    'all_features': all_features,
    'label_classes': list(label_encoder.classes_),
    'feature_descriptions': {
        'engagement_score': 'Weighted combo: attendance*0.6 + assignment*0.4',
        'performance_index': 'internal_marks - engagement_score',
        'multiple_risk_flags': 'Count of risk flags (0-3)'
    }
}
joblib.dump(feature_config, '../models/feature_config.joblib')
print("Saved: ../models/feature_config.joblib")

Saved: ../models/preprocessor.joblib
Saved: ../models/label_encoder.joblib
Saved: ../models/feature_config.joblib


In [10]:
# Save processed data for model training
train_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}
joblib.dump(train_data, '../data/processed/train_test_data.joblib')
print("\nSaved: ../data/processed/train_test_data.joblib")

# Also save as CSV for inspection
train_df = pd.DataFrame(X_train, columns=all_features)
train_df['risk_level'] = y_train
train_df.to_csv('../data/processed/train_processed.csv', index=False)
print("Saved: ../data/processed/train_processed.csv")


Saved: ../data/processed/train_test_data.joblib
Saved: ../data/processed/train_processed.csv


## 5. Preprocessing Summary

In [11]:
print("\n" + "="*60)
print("PREPROCESSING PIPELINE SUMMARY")
print("="*60)

print("\n1. Feature Engineering:")
for feat in engineered_features:
    print(f"   - {feat}: {feature_config['feature_descriptions'].get(feat, 'N/A')}")

print("\n2. Preprocessing Steps:")
print("   - StandardScaler: All numerical features normalized to mean=0, std=1")

print("\n3. Target Encoding:")
print(f"   - LabelEncoder: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")

print("\n4. Data Split:")
print(f"   - Train: {len(X_train)} samples")
print(f"   - Test: {len(X_test)} samples")

print("\n5. Saved Artifacts:")
print("   - preprocessor.joblib: ColumnTransformer with StandardScaler")
print("   - label_encoder.joblib: Risk level encoding")
print("   - feature_config.joblib: Feature metadata")
print("   - train_test_data.joblib: Preprocessed split data")

print("\n" + "="*60)
print("Next: Run 04_model_training.ipynb to train classifiers")
print("="*60)


PREPROCESSING PIPELINE SUMMARY

1. Feature Engineering:
   - engagement_score: Weighted combo: attendance*0.6 + assignment*0.4
   - performance_index: internal_marks - engagement_score
   - multiple_risk_flags: Count of risk flags (0-3)

2. Preprocessing Steps:
   - StandardScaler: All numerical features normalized to mean=0, std=1

3. Target Encoding:
   - LabelEncoder: {'High': 0, 'Low': 1, 'Moderate': 2}

4. Data Split:
   - Train: 400 samples
   - Test: 100 samples

5. Saved Artifacts:
   - preprocessor.joblib: ColumnTransformer with StandardScaler
   - label_encoder.joblib: Risk level encoding
   - feature_config.joblib: Feature metadata
   - train_test_data.joblib: Preprocessed split data

Next: Run 04_model_training.ipynb to train classifiers
